# Ukraine Procurement Explorer

EDA of 12.9M public procurement tenders from Prozorro system (2022-2025)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

INPUT_PATH = '/kaggle/input/prozorro-ukraine-procurement-2022-2025/'

## 1. Dataset Overview

In [ ]:
tenders = pd.concat([
    pd.read_csv(f'{INPUT_PATH}tenders_{year}.csv', low_memory=False)
    for year in [2022, 2023, 2024, 2025]
], ignore_index=True)

buyers = pd.read_csv(f'{INPUT_PATH}buyers.csv')
suppliers = pd.read_csv(f'{INPUT_PATH}suppliers.csv')

print(f'Total tenders: {len(tenders):,}')
print(f'Total buyers: {len(buyers):,}')
print(f'Total suppliers: {len(suppliers):,}')
print(f'Total value: {tenders["tender_value"].sum() / 1e12:.2f} trillion UAH')

## 2. Tenders by Year

In [ ]:
year_counts = tenders['year'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(year_counts.index.astype(str), year_counts.values, color='steelblue')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Tenders')
ax.set_title('Tenders by Year (2022-2025)')

for bar, val in zip(bars, year_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50000,
            f'{val/1e6:.2f}M', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Procurement Methods

In [ ]:
method_counts = tenders['procurement_method'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#3498db', '#2ecc71', '#e74c3c']
explode = (0, 0.1, 0.15)

wedges, texts, autotexts = axes[0].pie(
    method_counts.values,
    labels=method_counts.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    explode=explode,
    pctdistance=0.6
)
for autotext in autotexts:
    autotext.set_fontsize(11)
    autotext.set_fontweight('bold')
axes[0].set_title('Procurement Methods Distribution')

method_by_year = tenders.groupby(['year', 'procurement_method']).size().unstack(fill_value=0)
method_pct = method_by_year.div(method_by_year.sum(axis=1), axis=0) * 100

method_pct.plot(kind='bar', ax=axes[1], color=colors, width=0.8)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Percentage')
axes[1].set_title('Procurement Methods by Year')
axes[1].legend(title='Method')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

**Procurement methods:**
- **Limited** - Direct contracts without bidding (small amounts <50-200K UAH)
- **Open** - Competitive bidding (large procurements)
- **Selective** - Invited suppliers only

## 4. Seasonality (Q4 Effect)

In [ ]:
quarter_counts = tenders['quarter'].value_counts().sort_index()
expected = len(tenders) / 4

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(['Q1', 'Q2', 'Q3', 'Q4'], quarter_counts.values, color='steelblue')
ax.axhline(y=expected, color='red', linestyle='--', label='Expected (25%)')
ax.set_ylabel('Number of Tenders')
ax.set_title('Distribution by Quarter — Year-End Budget Spending Effect')
ax.legend()

bars[3].set_color('#e74c3c')

for bar, val in zip(bars, quarter_counts.values):
    pct = val / len(tenders) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50000,
            f'{pct:.1f}%', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

## 5. Competition Analysis

In [ ]:
competition_by_year = tenders.groupby('year').agg({
    'is_competitive': 'mean',
    'is_single_bidder': 'mean',
    'number_of_tenderers': 'mean'
}).round(4)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(competition_by_year.index, competition_by_year['is_single_bidder']*100,
             marker='o', linewidth=2, markersize=10, color='#e74c3c')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Single Bidder Rate (%)')
axes[0].set_title('Single Bidder Tenders')
axes[0].set_ylim(0, 10)

tenderers_dist = tenders['number_of_tenderers'].value_counts().sort_index().head(6)
axes[1].bar(tenderers_dist.index.astype(str), tenderers_dist.values, color='steelblue')
axes[1].set_xlabel('Number of Tenderers')
axes[1].set_ylabel('Number of Tenders')
axes[1].set_title('Distribution by Number of Tenderers')

plt.tight_layout()
plt.show()

## 6. Top Regions

In [ ]:
region_counts = buyers.groupby('buyer_region')['total_tenders'].sum().sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(range(len(region_counts)), region_counts.values, color='steelblue')
ax.set_yticks(range(len(region_counts)))
ax.set_yticklabels(region_counts.index)
ax.set_xlabel('Number of Tenders')
ax.set_title('Top 15 Regions by Number of Tenders')

bars[-1].set_color('#2ecc71')

for i, val in enumerate(region_counts.values):
    ax.text(val + 20000, i, f'{val/1e6:.2f}M', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 7. Value Distribution

In [ ]:
tender_values = tenders[tenders['tender_value'] > 0]['tender_value']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

log_values = np.log10(tender_values)
axes[0].hist(log_values, bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('log10(tender_value UAH)')
axes[0].set_ylabel('Count')
axes[0].set_title('Tender Value Distribution (log scale)')
axes[0].axvline(x=np.log10(50000), color='red', linestyle='--', alpha=0.7, label='50K threshold')
axes[0].legend()

def categorize_value(val):
    if val < 50000: return '<50K'
    elif val < 200000: return '50K-200K'
    elif val < 1000000: return '200K-1M'
    elif val < 10000000: return '1M-10M'
    else: return '>10M'

value_cats = tender_values.apply(categorize_value).value_counts()
order = ['<50K', '50K-200K', '200K-1M', '1M-10M', '>10M']
value_cats = value_cats.reindex(order)

colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c', '#9b59b6']
axes[1].bar(value_cats.index, value_cats.values, color=colors)
axes[1].set_xlabel('Value Category (UAH)')
axes[1].set_ylabel('Number of Tenders')
axes[1].set_title('Tenders by Value Category')

plt.tight_layout()
plt.show()

## 8. Risk Indicators

In [ ]:
indicators = {
    'Single Bidder': (tenders['is_single_bidder'].sum(), tenders['is_single_bidder'].mean()*100),
    'Weekend Tenders': (tenders['is_weekend'].sum(), tenders['is_weekend'].mean()*100),
    'Masked Buyers': (tenders['is_buyer_masked'].sum(), tenders['is_buyer_masked'].mean()*100),
    'Masked Suppliers': (tenders['is_supplier_masked'].sum(), tenders['is_supplier_masked'].mean()*100),
    'Unsuccessful Awards': (tenders['has_unsuccessful_awards'].sum(), tenders['has_unsuccessful_awards'].mean()*100),
    'Cancelled Awards': (tenders['has_cancelled_awards'].sum(), tenders['has_cancelled_awards'].mean()*100),
}

print('=' * 55)
print('RISK INDICATORS')
print('=' * 55)
print(f'{"Indicator":<25} {"Count":>12} {"Percentage":>12}')
print('-' * 55)
for name, (count, pct) in indicators.items():
    print(f'{name:<25} {count:>12,} {pct:>11.2f}%')
print('=' * 55)

## 9. CPV Categories — What Is Being Procured?

CPV (Common Procurement Vocabulary) is the EU standard classification for public procurement.

In [ ]:
cpv_names = {
    3: 'Agricultural products', 9: 'Fuel & energy', 14: 'Mining products',
    15: 'Food products', 18: 'Clothing & footwear', 22: 'Printed matter',
    24: 'Chemical products', 30: 'Office equipment', 31: 'Electrical equipment',
    32: 'Radio & telecom', 33: 'Medical equipment', 34: 'Transport',
    35: 'Security & defence', 37: 'Musical instruments', 38: 'Lab equipment',
    39: 'Furniture', 42: 'Industrial equipment', 44: 'Construction materials',
    45: 'Construction works', 48: 'Software', 50: 'Repair & maintenance',
    55: 'Hotel services', 60: 'Transport services', 63: 'Logistics',
    64: 'Postal & telecom services', 65: 'Utilities', 66: 'Financial services',
    70: 'Real estate', 71: 'Architecture & engineering', 72: 'IT services',
    73: 'R&D services', 75: 'Administrative services', 77: 'Agricultural services',
    79: 'Business services', 80: 'Education', 85: 'Healthcare',
    90: 'Sewage & environment', 92: 'Culture & sport', 98: 'Other services'
}

cpv_counts = tenders['main_cpv_2_digit'].value_counts().head(15)
cpv_values = tenders.groupby('main_cpv_2_digit')['tender_value'].sum().sort_values(ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

labels_count = [f"{cpv_names.get(c, f'CPV {c}')} ({c})" for c in cpv_counts.index]
bars1 = axes[0].barh(range(len(cpv_counts)), cpv_counts.values[::-1], color='steelblue')
axes[0].set_yticks(range(len(cpv_counts)))
axes[0].set_yticklabels(labels_count[::-1], fontsize=9)
axes[0].set_xlabel('Number of Tenders')
axes[0].set_title('Top 15 CPV Categories (by count)')

labels_value = [f"{cpv_names.get(c, f'CPV {c}')} ({c})" for c in cpv_values.index]
bars2 = axes[1].barh(range(len(cpv_values)), cpv_values.values[::-1] / 1e9, color='#2ecc71')
axes[1].set_yticks(range(len(cpv_values)))
axes[1].set_yticklabels(labels_value[::-1], fontsize=9)
axes[1].set_xlabel('Total Value (billion UAH)')
axes[1].set_title('Top 15 CPV Categories (by value)')

plt.tight_layout()
plt.show()

## 10. Top Suppliers

In [ ]:
top_by_awards = suppliers.nlargest(10, 'total_awards')[['supplier_name', 'supplier_region', 'total_awards', 'total_value']]
top_by_value = suppliers.nlargest(10, 'total_value')[['supplier_name', 'supplier_region', 'total_awards', 'total_value']]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

def truncate(name, max_len=35):
    return str(name)[:max_len] + '...' if len(str(name)) > max_len else str(name)

names_awards = [truncate(n) for n in top_by_awards['supplier_name']]
axes[0].barh(range(10), top_by_awards['total_awards'].values[::-1], color='steelblue')
axes[0].set_yticks(range(10))
axes[0].set_yticklabels(names_awards[::-1], fontsize=9)
axes[0].set_xlabel('Number of Won Tenders')
axes[0].set_title('Top 10 Suppliers (by count)')

names_value = [truncate(n) for n in top_by_value['supplier_name']]
axes[1].barh(range(10), top_by_value['total_value'].values[::-1] / 1e9, color='#2ecc71')
axes[1].set_yticks(range(10))
axes[1].set_yticklabels(names_value[::-1], fontsize=9)
axes[1].set_xlabel('Total Value (billion UAH)')
axes[1].set_title('Top 10 Suppliers (by value)')

plt.tight_layout()
plt.show()

## 11. Bids Analysis

The bids table contains all bid submissions from competitive tenders, including bid amounts and winner flags.

In [ ]:
bids = pd.concat([
    pd.read_csv(f'{INPUT_PATH}bids_{year}.csv')
    for year in [2022, 2023, 2024, 2025]
], ignore_index=True)

bidders = pd.read_csv(f'{INPUT_PATH}bidders.csv')

print(f'Total bids: {len(bids):,}')
print(f'Unique bidders: {len(bidders):,}')
print(f'Win rate (average): {bids["is_winner"].mean()*100:.1f}%')
print(f'\nBid status distribution:')
print(bids['bid_status'].value_counts().to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Bids per year
bids['bid_year'] = pd.to_datetime(bids['bid_date'], utc=True, errors='coerce').dt.year
bids_by_year = bids['bid_year'].value_counts().sort_index()
bids_by_year = bids_by_year[bids_by_year.index.isin([2022, 2023, 2024, 2025])]

axes[0].bar(bids_by_year.index.astype(str), bids_by_year.values, color='steelblue')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Number of Bids')
axes[0].set_title('Bids by Year')
for i, v in enumerate(bids_by_year.values):
    axes[0].text(i, v + 20000, f'{v/1e6:.2f}M', ha='center', fontweight='bold')

# 2. Bid status distribution
status_counts = bids['bid_status'].value_counts()
colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12', '#9b59b6']
axes[1].pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%',
            colors=colors[:len(status_counts)], startangle=90)
axes[1].set_title('Bid Status Distribution')

# 3. Bid amount distribution (log scale)
valid_amounts = bids[bids['bid_amount'] > 0]['bid_amount']
axes[2].hist(np.log10(valid_amounts), bins=50, color='steelblue', edgecolor='white')
axes[2].set_xlabel('log10(bid_amount UAH)')
axes[2].set_ylabel('Count')
axes[2].set_title('Bid Amount Distribution (log scale)')

plt.tight_layout()
plt.show()

In [ ]:
top_bidders = bidders.nlargest(10, 'total_bids')

wins_per_bidder = bids[bids['is_winner'] == 1].groupby('bidder_id').size().reset_index(name='wins_in_bids')
bidders_with_wins = bidders.merge(wins_per_bidder, on='bidder_id', how='left')
bidders_with_wins['wins_in_bids'] = bidders_with_wins['wins_in_bids'].fillna(0)
bidders_with_wins['win_rate'] = bidders_with_wins['wins_in_bids'] / bidders_with_wins['total_bids']

print(f'Total bidders: {len(bidders):,}')
print(f'Bidders with at least one win: {(bidders_with_wins["wins_in_bids"] > 0).sum():,} ({(bidders_with_wins["wins_in_bids"] > 0).mean()*100:.1f}%)')
print(f'Average win rate: {bidders_with_wins["win_rate"].mean()*100:.1f}%')
print(f'\nTop 10 most active bidders:')
print(top_bidders[['bidder_id', 'total_bids', 'unique_tenders', 'unique_buyers']].to_string(index=False))

## 12. Monthly Trends

In [ ]:
tenders['published_date'] = pd.to_datetime(tenders['published_date'], utc=True, errors='coerce')
tenders['month_period'] = tenders['published_date'].dt.to_period('M')

monthly = tenders.groupby('month_period').agg({
    'tender_id': 'count',
    'tender_value': 'sum'
}).reset_index()
monthly['month_period'] = monthly['month_period'].dt.to_timestamp()

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

axes[0].plot(monthly['month_period'], monthly['tender_id'] / 1000, 
             marker='o', markersize=4, linewidth=1.5, color='steelblue')
axes[0].axvline(pd.Timestamp('2022-02-24'), color='red', linestyle='--', alpha=0.7, label='War starts')
axes[0].fill_between(monthly['month_period'], monthly['tender_id'] / 1000, alpha=0.3)
axes[0].set_ylabel('Thousands of Tenders')
axes[0].set_title('Monthly Tender Count (2022-2025)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(monthly['month_period'], monthly['tender_value'] / 1e9,
             marker='o', markersize=4, linewidth=1.5, color='#2ecc71')
axes[1].axvline(pd.Timestamp('2022-02-24'), color='red', linestyle='--', alpha=0.7, label='War starts')
axes[1].fill_between(monthly['month_period'], monthly['tender_value'] / 1e9, alpha=0.3, color='#2ecc71')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Billion UAH')
axes[1].set_title('Monthly Tender Value (2022-2025)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

**Key findings:**

1. **Scale**: 12.9M tenders, 3.82 trillion UAH over 4 years
2. **Methods**: 90%+ limited (direct contracts), only ~5-9% open (competitive)
3. **Seasonality**: Q4 has more tenders (year-end budget spending effect)
4. **Competition**: 93%+ tenders have no competition (0 tenderers), only 3-5% competitive
5. **Geography**: Kyiv region leads, followed by Lviv, Odesa, Poltava
6. **CPV**: Most tenders — medical equipment & food; highest value — fuel & construction
7. **Suppliers**: Top-10 suppliers dominate the market
8. **War impact**: Drop in March 2022, then recovery and growth